# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1 — World choropleth: life expectancy vs the global average (diverging)
# ── Data prep ─────────────────────────────────────────────────────────────────
gm_2007 = px.data.gapminder().query("year == 2007").copy()

world_avg = gm_2007['lifeExp'].mean()                 # global mean life expectancy
gm_2007['deviation'] = gm_2007['lifeExp'] - world_avg # +ve above world avg, -ve below

# Symmetric range so 0 (world average) sits exactly at the white midpoint
lim = gm_2007['deviation'].abs().max()

# ── Choropleth — DIVERGING scale centred at zero ──────────────────────────────
# Why px.choropleth (not choropleth_map): built-in ISO-3 country geometry, no
# external GeoJSON needed, and the orthographic/natural-earth view suits a global
# above/below comparison. Why DIVERGING (RdBu): the variable has a meaningful
# midpoint (the world average) and we care about direction either side of it.
fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',                # ISO-3 codes → built-in geometry
    color='deviation',
    color_continuous_scale='RdBu',
    range_color=[-lim, lim],              # forces 0 to the diverging midpoint
    color_continuous_midpoint=0,
    hover_name='country',
    hover_data={
        'iso_alpha': False,
        'lifeExp': ':.1f',                # raw life expectancy
        'deviation': ':+.1f',             # signed deviation from world average
    },
    labels={'lifeExp': 'Life expectancy (yrs)', 'deviation': 'vs world avg (yrs)'},
    title=f'Sub-Saharan Africa sits furthest below the world average '
          f'({world_avg:.0f} yrs) \u2014 the deepest reds on the map',
)

fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=15), x=0.01, xanchor='left'),
    margin=dict(l=0, r=0, t=55, b=0),
    paper_bgcolor='white',
    coloraxis_colorbar=dict(title='Years vs<br>world avg'),
    height=560,
)
fig.show()


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** US state boundaries from the Plotly datasets repo —
`https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json`
is county-level; for clean state polygons I use the public
`https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json`
file, whose `properties.name` holds the full state name.

**Chart type chosen** (`px.choropleth` or `px.choropleth_map`) **and reason:**

`px.choropleth`. The data is a complete administrative partition (every US state),
not a scatter of points over terrain, so I don't need an interactive tiled
basemap underneath. A plain GeoJSON-backed choropleth with `scope='usa'` renders
faster, has no tile dependency, and keeps the focus on the polygons themselves.

**Colour scale chosen** (sequential or diverging) **and reason:**

Sequential (`YlOrRd`). The variable — population per state — is a single
non-negative quantity with no meaningful midpoint. A sequential ramp maps "more"
to "darker" intuitively; a diverging scale would imply a centre point the data
doesn't have.

In [ ]:
# Task 2 — Choropleth from a self-sourced GeoJSON (US states)
import urllib.request

# ── Load a GeoJSON found online (US state polygons) ───────────────────────────
GEOJSON_URL = ('https://raw.githubusercontent.com/PublicaMundi/MappingAPI/'
               'master/data/geojson/us-states.json')
with urllib.request.urlopen(GEOJSON_URL) as resp:
    us_states = json.load(resp)

# ALWAYS inspect properties first to find the matching featureidkey
print('Feature properties:', us_states['features'][0]['properties'])
# -> {'name': 'Alabama', 'density': ...}  → the key that matches our data is 'name'

# ── Matching dataset: 2020 US Census resident population by state (millions) ───
pop = {
    'Alabama':5.02,'Alaska':0.73,'Arizona':7.15,'Arkansas':3.01,'California':39.54,
    'Colorado':5.77,'Connecticut':3.61,'Delaware':0.99,'District of Columbia':0.69,
    'Florida':21.54,'Georgia':10.71,'Hawaii':1.46,'Idaho':1.84,'Illinois':12.81,
    'Indiana':6.79,'Iowa':3.19,'Kansas':2.94,'Kentucky':4.51,'Louisiana':4.66,
    'Maine':1.36,'Maryland':6.18,'Massachusetts':7.03,'Michigan':10.08,
    'Minnesota':5.71,'Mississippi':2.96,'Missouri':6.15,'Montana':1.08,
    'Nebraska':1.96,'Nevada':3.10,'New Hampshire':1.38,'New Jersey':9.29,
    'New Mexico':2.12,'New York':20.20,'North Carolina':10.44,'North Dakota':0.78,
    'Ohio':11.80,'Oklahoma':3.96,'Oregon':4.24,'Pennsylvania':13.00,
    'Rhode Island':1.10,'South Carolina':5.12,'South Dakota':0.89,
    'Tennessee':6.91,'Texas':29.15,'Utah':3.27,'Vermont':0.64,'Virginia':8.63,
    'Washington':7.71,'West Virginia':1.79,'Wisconsin':5.89,'Wyoming':0.58,
}
pop_df = pd.DataFrame({'state': list(pop), 'population_m': list(pop.values())})

# ── Choropleth — SEQUENTIAL scale (YlOrRd) ────────────────────────────────────
fig = px.choropleth(
    pop_df,
    geojson=us_states,
    locations='state',
    featureidkey='properties.name',       # matched by inspecting the GeoJSON above
    color='population_m',
    color_continuous_scale='YlOrRd',
    scope='usa',
    hover_name='state',
    labels={'population_m': 'Population (millions)'},
    title='California, Texas and Florida hold the population gravity '
          '\u2014 the Sun Belt and coasts dwarf the Mountain West',
)

fig.update_geos(showlakes=False)
fig.update_layout(
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=15), x=0.01, xanchor='left'),
    margin=dict(l=0, r=0, t=55, b=0),
    paper_bgcolor='white',
    coloraxis_colorbar=dict(title='Pop.<br>(millions)'),
    height=560,
)
fig.show()
